## 0. Importação das biliotecas

In [ ]:
import yaml
import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Float, DateTime, Boolean


## 1. Funções

In [2]:
def tratar_csv(caminho, separador=';'):
    df = pd.read_csv(caminho, sep=separador)
    
    # Remover caracteres esquisitos no início do arquivo
    df.columns = df.columns.str.replace('ï»¿', '')
    
    # Padronizar nomes das colunas (minusculas, sem espaços)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    return df

## 1. Importação dos Dados

In [3]:
# Lendo os arquivos 
df_cliente = tratar_csv('./data/cliente.csv')
df_volumetria = tratar_csv('./data/volumetria.csv')
df_cnae = tratar_csv('./data/cliente_cnae.csv')

In [4]:
df_cnae.head()

,cli_codigo,cnae,cnae_descr,cne_cnae_principal
0,72613,7120-1/00,Testes e análises técnicas,1
1,72613,7490-1/99,"Outras atividades profissionais, científicas e...",0
2,72613,7210-0/00,Pesquisa e desenvolvimento experimental em ciê...,0
3,281372,1340-5/01,"Estamparia e texturização em fios, tecidos, ar...",1
4,281372,1311-1/00,Preparação e fiação de fibras de algodão,0


In [5]:
df_cliente.head()

,cli_codigo,razao,cli_fisjur,loc_codibge,loc_descricao,uf_sigla
0,4928843,CLIENTE - 4928843,F,4215000,RIO NEGRINHO,SC
1,1970,CLIENTE - 1970,F,4306809,ENCANTADO,RS
2,2187,CLIENTE - 2187,F,4321501,TORRES,RS
3,2646,CLIENTE - 2646,F,4312625,MULITERNO,RS
4,4670349,CLIENTE - 4670349,F,4114302,MANDIRITUBA,PR


In [6]:
df_volumetria.head()

,via_codigo,dt_viagem,cli_codigo,tipopesocubico,qtd_ctes,volumes,peso,m3,perc_descm3,pesom3
0,1395158,2026-01-05 20:32:00.000,3057201,M,1,6,115,"1,12",0,1.120
1,1395158,2026-01-05 20:32:00.000,3412661,P,1,5,"27,38",0,0,27.380
2,1395158,2026-01-05 20:32:00.000,3690638,P,1,9,"30,924",0,0,30.924
3,1395158,2026-01-05 20:32:00.000,4295681,M,1,1,151,"2,61",0,2.610
4,1396958,2026-01-05 07:00:00.000,2553883,P,1,1,"0,001",0,0,0.001


## 2. Qualidade dos Dados

### 2.1 Volumetria dos dados

In [7]:
# Total de registros e coluns para cada tabela
print(f'Total de {df_cliente.shape[0]} registros e {df_cliente.shape[1]} colunas na tabela df_clientes\n')
print(f'Total de {df_volumetria.shape[0]} registros e {df_volumetria.shape[1]} colunas na tabela df_volumetria\n')
print(f'Total de {df_cnae.shape[0]} registros e {df_cnae.shape[1]} colunas na tabela df_cnae\n')


Total de 131728 registros e 6 colunas na tabela df_clientes

Total de 1048575 registros e 10 colunas na tabela df_volumetria

Total de 764658 registros e 4 colunas na tabela df_cnae



### 2.2 Tipos dos Dados nas tabelas

In [8]:
# Verificando tipo de dados
print(f'Tabela Clientes:\n{df_cliente.dtypes}\n')
print(f'Tabela Volumetria:\n{df_volumetria.dtypes}\n')
print(f'Tabela CNAE:\n{df_cnae.dtypes}\n')

Tabela Clientes:
cli_codigo        int64
razao            object
cli_fisjur       object
loc_codibge       int64
loc_descricao    object
uf_sigla         object
dtype: object

Tabela Volumetria:
via_codigo          int64
dt_viagem          object
cli_codigo          int64
tipopesocubico     object
qtd_ctes            int64
volumes             int64
peso               object
m3                 object
perc_descm3        object
pesom3            float64
dtype: object

Tabela CNAE:
cli_codigo             int64
cnae                  object
cnae_descr            object
cne_cnae_principal     int64
dtype: object



### 2.3 Verificando dados nulos

In [9]:
print(f'Qtd de valores nulos na tabela Clientes:\n{df_cliente.isna().sum()}')
print(f'\nQtd de valores nulos na tabela Volumetria:\n{df_volumetria.isna().sum()}')
print(f'\nQtd de valores nulos na tabela CNAE:\n{df_cnae.isna().sum()}')

Qtd de valores nulos na tabela Clientes:
cli_codigo       0
razao            0
cli_fisjur       0
loc_codibge      0
loc_descricao    0
uf_sigla         0
dtype: int64

Qtd de valores nulos na tabela Volumetria:
via_codigo        0
dt_viagem         0
cli_codigo        0
tipopesocubico    0
qtd_ctes          0
volumes           0
peso              0
m3                0
perc_descm3       0
pesom3            0
dtype: int64

Qtd de valores nulos na tabela CNAE:
cli_codigo            0
cnae                  0
cnae_descr            0
cne_cnae_principal    0
dtype: int64


### 2.4 Duplicidade nas tabelas

In [10]:
print(f'Quantidade de registros duplicados nas tabelas:\n')
print(f'- df_volumetria: {df_volumetria.duplicated().sum()}')
print(f'- df_cliente: {df_cliente.duplicated().sum()}')
print(f'- df_cnae: {df_cnae.duplicated().sum()}')

Quantidade de registros duplicados nas tabelas:

- df_volumetria: 0
- df_cliente: 0
- df_cnae: 0


### 2.5 Valores distintos para colunas categóricas cahve

In [11]:
col_cli = ['cli_fisjur', 'uf_sigla']

print(f'Valores únicos da tabela df_cliente:\n')
for col in col_cli:
    print(f'- coluna {col}:\n{df_cliente[col].unique()}\n')

print(f'Valores únicos da tabela df_cnae:\n')
print(f'- coluna cne_cnae_principal:\n{df_cnae['cne_cnae_principal'].unique()}\n')

print(f'Valores únicos da tabela df_volumetria:\n')
print(f'- coluna tipopesocubico:\n{df_volumetria["tipopesocubico"].unique()}\n')


Valores únicos da tabela df_cliente:

- coluna cli_fisjur:
['F' 'J']

- coluna uf_sigla:
['SC' 'RS' 'PR' 'SP' 'CE' 'MT' 'MS' 'AM' 'AL' 'MG' 'TO' 'MA' 'GO' 'RO'
 'PA' 'RJ' 'BA' 'PE' 'ES' 'SE' 'PI' 'RN' 'RR' 'DF' 'PB']

Valores únicos da tabela df_cnae:

- coluna cne_cnae_principal:
[1 0]

Valores únicos da tabela df_volumetria:

- coluna tipopesocubico:
['M' 'P']



## 3. Tratamento dos Dados

- Tipos de Dados:
    - Tabelas Cliente e Cliente_Cnae não precisam de tratamento para tipo de dados
    - Tabela Volumetria precisa realizar os seguintes tratamentos de tipo de dados:
        - Coluna dt_viagem: de str -> datatime
        - Coluna peso: de str -> float
        - Coluna m3: de str -> float
        - Coluna perc_descm3: de str -> float
- Dados Nulos:
    - Nenhuma tabela apresenta dados nulos

- Duplicidade dos Dados:
    - Nenhuma tabela apresenta dados duplicados

- Dados distintos para colunas categóricas chave:
    - Todas as colunas apresentaram dados concistentes

### 3.1 Tratamento tabela Volumetria

In [12]:
df_volumetria.head()

,via_codigo,dt_viagem,cli_codigo,tipopesocubico,qtd_ctes,volumes,peso,m3,perc_descm3,pesom3
0,1395158,2026-01-05 20:32:00.000,3057201,M,1,6,115,"1,12",0,1.120
1,1395158,2026-01-05 20:32:00.000,3412661,P,1,5,"27,38",0,0,27.380
2,1395158,2026-01-05 20:32:00.000,3690638,P,1,9,"30,924",0,0,30.924
3,1395158,2026-01-05 20:32:00.000,4295681,M,1,1,151,"2,61",0,2.610
4,1396958,2026-01-05 07:00:00.000,2553883,P,1,1,"0,001",0,0,0.001


In [13]:
# Tratamento da coluna dt_viagem para o tipo datetime
df_volumetria['dt_viagem'] = pd.to_datetime(df_volumetria['dt_viagem'], errors='coerce')

# Verificando o resultado do tratamento
print(f'Tipo da coluna dt_viagem após tratamento:\n{df_volumetria["dt_viagem"].dtypes}\n')
df_volumetria.head()


Tipo da coluna dt_viagem após tratamento:
datetime64[ns]



,via_codigo,dt_viagem,cli_codigo,tipopesocubico,qtd_ctes,volumes,peso,m3,perc_descm3,pesom3
0,1395158,2026-01-05 20:32:00,3057201,M,1,6,115,"1,12",0,1.120
1,1395158,2026-01-05 20:32:00,3412661,P,1,5,"27,38",0,0,27.380
2,1395158,2026-01-05 20:32:00,3690638,P,1,9,"30,924",0,0,30.924
3,1395158,2026-01-05 20:32:00,4295681,M,1,1,151,"2,61",0,2.610
4,1396958,2026-01-05 07:00:00,2553883,P,1,1,"0,001",0,0,0.001


In [14]:
# tratamento da coluna peso	m3	perc_descm3 para o tipo float
cols_to_convert = ['peso', 'm3', 'perc_descm3']
for col in cols_to_convert:
    df_volumetria[col] = df_volumetria[col].str.replace(',', '.').astype(float)
print(f'Tipos das colunas peso, m3 e perc_descm3 após tratamento:\n{df_volumetria[cols_to_convert].dtypes}\n')

Tipos das colunas peso, m3 e perc_descm3 após tratamento:
peso           float64
m3             float64
perc_descm3    float64
dtype: object



In [15]:
df_volumetria.head()

,via_codigo,dt_viagem,cli_codigo,tipopesocubico,qtd_ctes,volumes,peso,m3,perc_descm3,pesom3
0,1395158,2026-01-05 20:32:00,3057201,M,1,6,115.000,1.12,0.0,1.120
1,1395158,2026-01-05 20:32:00,3412661,P,1,5,27.380,0.00,0.0,27.380
2,1395158,2026-01-05 20:32:00,3690638,P,1,9,30.924,0.00,0.0,30.924
3,1395158,2026-01-05 20:32:00,4295681,M,1,1,151.000,2.61,0.0,2.610
4,1396958,2026-01-05 07:00:00,2553883,P,1,1,0.001,0.00,0.0,0.001


## 4. Inserção dos Dados Tratados no Banco de Dados Postgres no Rander

### 4.1 Conexão com Banco

In [16]:
# Carregar credenciais
with open('./config/database_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# String de conexão
conn_string = f"postgresql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}"

try:
    # Testar conexão
    engine = create_engine(conn_string)
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 as test"))
        print("conectado!")
        
        # Testar query simples
        result = conn.execute(text("SELECT version();"))
        version = result.fetchone()
        print(f"PostgreSQL: {version[0][:50]}...")
        
except Exception as e:
    print(f"ERRO: {e}")

conectado!
PostgreSQL: PostgreSQL 17.7 (Debian 17.7-3.pgdg12+1) on x86_64...


### 4.2 Inserção dos Dados no Banco

In [53]:
# 1. Dimensão Cliente
df_cliente.to_sql(
    'dim_cliente', 
    engine, 
    index=False, 
    if_exists='replace', 
    dtype={
        'cli_codigo': Integer(),
        'razao': String(),
        'cli_fisjur': String(1),
        'loc_codibge': Integer(),
        'loc_descricao': String(),
        'uf_sigla': String(2)
    }
)

# 2. Dimensão CNAE
df_cnae.to_sql(
    'dim_cliente_cnae', 
    engine, 
    index=False, 
    if_exists='replace', 
    dtype={
        'cli_codigo': Integer(),
        'cnae': String(),
        'cnae_descr': String(),
        'cne_cnae_principal': Integer()
    }
)

# 3. Tabela Fato Volumetria
df_volumetria.to_sql(
    'fato_volumetria', 
    engine, 
    index=False, 
    if_exists='replace', 
    dtype={
        'via_codigo': Integer(),
        'dt_viagem': DateTime(),
        'cli_codigo': Integer(),
        'tipopesocubico': String(1),
        'qtd_ctes': Integer(),
        'volumes': Integer(),
        'peso': Float(),
        'm3': Float(),
        'perc_descm3': Float(),
        'pesom3': Float()
    }
)

print("Dados inseridos com sucesso!")

#Criaando chaves primárias no banco
with engine.connect() as conn:
    try:
        conn.execute(text("ALTER TABLE dim_cliente ADD PRIMARY KEY (cli_codigo);"))
        conn.execute(text("CREATE INDEX idx_volumetria_cliente ON fato_volumetria(cli_codigo);"))
        conn.commit()
        print("Chaves e índices criados.")
    except Exception as e:
        print(f"Erro ao criar chaves: {e}")

Dados inseridos com sucesso!
Chaves e índices criados.
